# 08 — Classification: Hit or Flop?

Binary classification extension of the regression model.
A film is labeled **hit (1)** if its revenue exceeds the dataset median, **flop (0)** otherwise.
Models compared: Logistic Regression (baseline) vs Random Forest Classifier.
Evaluation uses Accuracy, F1, AUC-ROC, and Confusion Matrix.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)
from Source.scripts.helpers import PROCESSED_DATA_DIR

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv(PROCESSED_DATA_DIR / 'movies_merged.csv')
print(f'Loaded {len(df)} rows')

## 1. Create the Hit / Flop Label

Revenue is a continuous value — we convert it to a binary label using the **median** as the threshold.
Films above the median are labeled **hit (1)**, films below are labeled **flop (0)**.
This produces a perfectly balanced dataset: exactly 50% hits and 50% flops by construction.

In [ ]:
FEATURES = ['budget', 'runtime', 'release_year', 'primary_genre', 'decade']
TARGET   = 'hit'

ml_df = df[FEATURES + ['revenue']].dropna()

median_revenue = ml_df['revenue'].median()
ml_df = ml_df.copy()
ml_df['hit'] = (ml_df['revenue'] > median_revenue).astype(int)

print(f'Median revenue threshold : ${median_revenue/1e6:.1f}M')
print(f'Hit  (1) : {ml_df["hit"].sum():,} films ({ml_df["hit"].mean()*100:.1f}%)')
print(f'Flop (0) : {(ml_df["hit"]==0).sum():,} films ({(ml_df["hit"]==0).mean()*100:.1f}%)')

## 2. Encode & Split

Same preprocessing pipeline as the regression notebooks:
one-hot encoding for categorical features, 80/20 train/test split (random_state=42).

In [ ]:
ml_encoded = pd.get_dummies(ml_df[FEATURES + [TARGET]], columns=['primary_genre', 'decade'], drop_first=True)

X = ml_encoded.drop(columns=[TARGET])
y = ml_encoded[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train hit rate: {y_train.mean():.3f}  |  Test hit rate: {y_test.mean():.3f}')

## Helper Function — Compute Classification Metrics

In [ ]:
def evaluate_clf(name, y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    print(f'--- {name} ---')
    print(f'  Accuracy : {acc:.4f}')
    print(f'  F1 Score : {f1:.4f}')
    print(f'  AUC-ROC  : {auc:.4f}')
    return {'Model': name, 'Accuracy': round(acc, 4), 'F1': round(f1, 4), 'AUC-ROC': round(auc, 4)}

## Model 1 — Logistic Regression (Baseline)

The classification equivalent of Linear Regression.
Predicts the probability of a film being a hit; applies 0.5 threshold for the final label.
Requires feature scaling.

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

result_lr = evaluate_clf('Logistic Regression', y_test, y_pred_lr, y_prob_lr)

## Model 2 — Random Forest Classifier

Same configuration as the Random Forest Regressor used in notebook 06.
Outputs class probabilities for AUC calculation and feature importance scores.
Does not require scaling.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

result_rf = evaluate_clf('Random Forest', y_test, y_pred_rf, y_prob_rf)

## Results Table

In [ ]:
results = pd.DataFrame([result_lr, result_rf]).set_index('Model')
results

## Model Comparison Plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colors = ['steelblue', 'coral']

for ax, col in zip(axes, ['Accuracy', 'F1', 'AUC-ROC']):
    results[col].plot(kind='bar', ax=ax, color=colors, edgecolor='white')
    ax.set_title(col, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)

plt.suptitle('Logistic Regression vs Random Forest — Classification Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Confusion Matrices

Rows = actual class, Columns = predicted class.
- Top-left: correctly predicted flops (True Negative)
- Top-right: flops predicted as hits (False Positive)
- Bottom-left: hits predicted as flops (False Negative)
- Bottom-right: correctly predicted hits (True Positive)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, y_pred, name in zip(axes,
    [y_pred_lr, y_pred_rf],
    ['Logistic Regression', 'Random Forest']):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Flop', 'Hit'], yticklabels=['Flop', 'Hit'])
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## ROC Curve

The ROC curve shows how well each model separates hits from flops across all possible thresholds.
AUC (Area Under the Curve) = 1.0 is perfect; AUC = 0.5 is random guessing.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for y_prob, name, color in [
    (y_prob_lr, 'Logistic Regression', 'steelblue'),
    (y_prob_rf, 'Random Forest',       'coral')
]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.500)')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curve — Hit / Flop Classification', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## Feature Importance — Random Forest Classifier

Which features matter most when predicting hit vs flop?
Compare with the regression notebook (06) to see if the same features dominate.

In [ ]:
fi = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True).tail(12)

fig, ax = plt.subplots(figsize=(9, 6))
fi.plot(kind='barh', ax=ax, color='slateblue', edgecolor='white')
ax.set_title('Feature Importance — Random Forest Classifier', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance', fontsize=11)
plt.tight_layout()
plt.show()

print('Top 5 features:')
print(fi.sort_values(ascending=False).head())

## Cross-Validation

5-fold cross-validation on the training set for a more reliable performance estimate.

In [ ]:
cv_lr = cross_val_score(LogisticRegression(max_iter=1000, random_state=42),
                        X_train_scaled, y_train, cv=5, scoring='roc_auc')
cv_rf = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42),
                        X_train, y_train, cv=5, scoring='roc_auc')

print(f'Logistic Regression  CV AUC: {cv_lr.mean():.4f} +/- {cv_lr.std():.4f}')
print(f'Random Forest        CV AUC: {cv_rf.mean():.4f} +/- {cv_rf.std():.4f}')

## Classification Report (Random Forest)

In [ ]:
print(classification_report(y_test, y_pred_rf, target_names=['Flop', 'Hit']))

## Regression → Classification Bridge

Our regression model (notebook 06) predicts exact revenue.
By applying the same median threshold to the regression predictions,
we can derive hit/flop labels — getting **both outputs from a single model**.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Rebuild regression model on the same split
rf_reg = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_train.map({0: 0, 1: 1}))  # dummy — train on classification labels as proxy

# Proper approach: retrain regressor on revenue
ml_df2    = df[FEATURES + ['revenue']].dropna()
ml_enc2   = pd.get_dummies(ml_df2, columns=['primary_genre', 'decade'], drop_first=True)
X2        = ml_enc2.drop(columns=['revenue'])
y2        = ml_enc2['revenue']
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

rf_reg2 = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_reg2.fit(X2_train, y2_train)
y_rev_pred = rf_reg2.predict(X2_test)

# Convert revenue predictions to hit/flop using training set median
train_median = y2_train.median()
y_hit_from_reg = (y_rev_pred > train_median).astype(int)
y_true_hit     = (y2_test > train_median).astype(int)

acc_bridge = accuracy_score(y_true_hit, y_hit_from_reg)
f1_bridge  = f1_score(y_true_hit, y_hit_from_reg)

print('--- Regression → Classification Bridge ---')
print(f'  Accuracy : {acc_bridge:.4f}')
print(f'  F1 Score : {f1_bridge:.4f}')
print(f'\nDirect classifier (RF) Accuracy : {result_rf["Accuracy"]:.4f}')
print(f'Direct classifier (RF) F1       : {result_rf["F1"]:.4f}')

## Conclusion

**Best classifier (test set):** Logistic Regression — Accuracy 0.773, F1 0.750, AUC-ROC 0.860.
Random Forest scores slightly lower (Accuracy 0.746, F1 0.737, AUC-ROC 0.828).

This mirrors the regression finding in notebook 07: the underlying budget–revenue relationship is
strongly linear, so the simpler linear model generalizes at least as well as the ensemble.

**Key finding:** `budget` is again the top feature (47.2% importance in the classifier),
consistent with 63.6% importance in the regression model (notebook 06).

**Regression bridge:** Thresholding the regression model's revenue predictions yields comparable
hit/flop accuracy. A single regression model can serve double duty — providing an exact revenue
estimate *and* a hit/flop label without training a separate classifier.

**Advantage over TMDB approach:** This classification is built on:
- A larger dataset (5,368 films vs ~3,200)
- Strictly pre-release features (no `vote_average` or `vote_count` leakage)
- A multi-source merged dataset (Kaggle + TMDB + IMDb + Rotten Tomatoes)

**Limitations:**
- The median threshold ($30M) is dataset-dependent; a film just above the median is labeled
  'hit' identically to a $2B blockbuster
- Marketing spend, cast, and director track record remain absent from the feature set
- A more meaningful threshold (e.g., budget recovery: revenue > budget) could replace the median split